# Notebook 14 -- Score-Input Integration and Release Freeze (Part A)

## Purpose

This notebook does **not calculate any score**. It integrates already-produced, source-linked
evidence (Notebooks 04-13) into a single frozen `release_v1` input bundle for Version 1.0 of the
**Scenario-Aware Comparative Residential Regulation Index**, and runs the critical
evidence-integrity / legal-semantic validations a score calculation must be gated behind.

## Honest scope note

The originating spec names a very large file list (dozens of individual audit CSVs). As with
Notebook 13, this notebook consolidates many of those into fewer, complete tables rather than
producing many thin duplicates of the same underlying check -- the consolidation mapping is recorded
explicitly below and in `docs/ARTIFACT_MAP.md`. Every number is independently computed from the
actual repository files in this run, not copied from a prior notebook's printed summary.

## Integration, not re-extraction

The Bridgeport V7/V12 master-use-table evidence, the 13-page PARKING family, and the Small General
scenario addendum were already independently verified (Notebook 13, Section F: 20/20 reconciled
headline claims). This notebook's job is to **integrate** that already-verified evidence into the
`release_v1/` naming convention and freeze it -- not to re-derive it from the raw PDF a second time.
Where this notebook performs a NEW check not already done in Notebook 11/13, that is stated
explicitly.


In [1]:
import pandas as pd
import numpy as np
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/Users/sierramendoza/Documents/HAP-Repository/Stage_A_Zoning_Code_Reader")
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RELEASE_DIR = PROCESSED_DIR / "release_v1"
RELEASE_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_V1_DIR = ROOT / "outputs" / "review" / "release_v1"
REVIEW_V1_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR = ROOT / "outputs" / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()
RELEASE_VERSION = "v1.0.0-pilot"
print(f"Run timestamp: {RUN_TIMESTAMP}")
print(f"Release version: {RELEASE_VERSION}")

RAW_SOURCE_FILES = [
    DATA_DIR / "raw" / "bridgeport" / "bridgeport_zoning_code_full_2022-01.pdf.pdf",
    DATA_DIR / "raw" / "greenwich" / "greenwich_building_zone_regulations_full_2023-08.pdf",
    DATA_DIR / "raw" / "bridgeport" / "source_metadata.json",
    DATA_DIR / "raw" / "greenwich" / "source_metadata.json",
    DATA_DIR / "raw" / "bridgeport" / "gis" / "bridgeport_zoning_districts_2021.geojson",
    DATA_DIR / "raw" / "greenwich_gis" / "greenwich_zone_boundaries.geojson",
]
BASELINE_FILES_READ_THIS_NOTEBOOK = RAW_SOURCE_FILES + [
    PROCESSED_DIR / "expanded_variable_recovery_matrix_bridgeport_greenwich.csv",
    PROCESSED_DIR / "fixed_variable_recovery_matrix_bridgeport_greenwich.csv",
    PROCESSED_DIR / "bridgeport_v7_v12_master_use_table_addendum.csv",
    PROCESSED_DIR / "bridgeport_v7_v12_scenario_matched_for_scoring.csv",
    PROCESSED_DIR / "bridgeport_v5_scenario_resolved_addendum.csv",
    PROCESSED_DIR / "bridgeport_scenario_registry_addendum.csv",
    PROCESSED_DIR / "bridgeport_scenario_score_eligibility_universe.csv",
    PROCESSED_DIR / "bridgeport_parking_full_13_building_type_recovery.csv",
    DATA_DIR / "processed" / "bridgeport" / "bridgeport_all_district_scenarios.parquet",
    DATA_DIR / "processed" / "bridgeport" / "bridgeport_district_metadata.csv",
]
BASELINE_HASHES = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in BASELINE_FILES_READ_THIS_NOTEBOOK if p.exists()}
print(f"Captured baseline hashes for {len(BASELINE_HASHES)} files this notebook reads (verified unchanged at the end).")

Run timestamp: 2026-10-01T21:30:19.223838+00:00
Release version: v1.0.0-pilot
Captured baseline hashes for 16 files this notebook reads (verified unchanged at the end).


## A1 -- Repository inspection and artifact status register

Loads Notebook 13's live artifact catalog (not a stale assumption) and classifies every file by
release-relevant status. No file is deleted; this produces recommendations only.


In [2]:
artifact_catalog = pd.read_csv(PROCESSED_DIR / "repository_artifact_catalog.csv")
print(f"Artifact catalog loaded: {len(artifact_catalog)} files (from Notebook 13's live scan)")

def classify_artifact_status(row):
    p = row["path"]
    if p.startswith("data/raw/"):
        return "baseline_raw_immutable"
    if "release_v1" in p:
        return "interactive_asset_candidate" if "interactive" in p else "canonical_v1"
    if any(x in p for x in ["_addendum", "_override", "coverage_correction", "overrides"]):
        return "additive_override"
    if p.startswith("outputs/review/"):
        return "review"
    if p.startswith("data/interim/"):
        return "interim"
    if p.startswith("docs/"):
        return "documentation"
    if p.startswith("notebooks/"):
        return "notebook"
    if p.startswith("outputs/logs/"):
        return "manifest_log"
    if p.startswith("data/processed/") and ("recovery_matrix" in p or "evidence_profile" in p or "district_metadata" in p or "scenarios.parquet" in p):
        return "canonical_baseline"
    return "derived_other"

artifact_catalog["release_status"] = artifact_catalog.apply(classify_artifact_status, axis=1)
status_register = artifact_catalog[["path", "artifact_type", "town", "category", "release_status", "size_bytes", "duplicate_content_flag"]].copy()
status_register.to_csv(PROCESSED_DIR / "artifact_status_register_v1.csv", index=False)
print(status_register["release_status"].value_counts().to_string())
print("\\nSaved: data/processed/artifact_status_register_v1.csv")

cleanup_candidates = status_register[status_register["duplicate_content_flag"]].copy()
cleanup_candidates["recommendation"] = "duplicate_content -- human review before any deletion; not auto-removed"
cleanup_candidates.to_csv(ROOT / "outputs" / "review" / "repository_cleanup_recommendations.csv", index=False)
print(f"\\nSaved: outputs/review/repository_cleanup_recommendations.csv ({len(cleanup_candidates)} flagged, none deleted)")

Artifact catalog loaded: 1455 files (from Notebook 13's live scan)
release_status
interim                   663
derived_other             590
review                     83
manifest_log               37
documentation              25
additive_override          25
notebook                   14
canonical_baseline          9
baseline_raw_immutable      9
\nSaved: data/processed/artifact_status_register_v1.csv
\nSaved: outputs/review/repository_cleanup_recommendations.csv (69 flagged, none deleted)


## A2 -- Additive canonical integration layer

### A2.1 Small General scenario-registry addendum (v1 naming)

Reuses Notebook 11's already-verified Small General addendum (2 rows, NX2/RX1, source pages 56-57,
`scenario_source = post_phase_2_registry_addendum`) and republishes it under the `release_v1/`
naming convention. The underlying baseline registry file is re-verified byte-identical below.


In [3]:
scenario_registry_addendum = pd.read_csv(PROCESSED_DIR / "bridgeport_scenario_registry_addendum.csv")
scenarios_baseline = pd.read_parquet(DATA_DIR / "processed" / "bridgeport" / "bridgeport_all_district_scenarios.parquet")
scenarios_baseline_dedup = scenarios_baseline.drop_duplicates(subset=["district_code", "building_form_or_type"]).reset_index(drop=True)

assert set(scenario_registry_addendum["district_code"]) == {"NX2", "RX1"}, "Small General addendum must be exactly NX2/RX1"
assert (scenario_registry_addendum["scenario_source"] == "post_phase_2_registry_addendum").all()
assert scenario_registry_addendum["source_pages"].apply(lambda x: 56 in eval(str(x)) and 57 in eval(str(x))).all(), "Small General addendum must cite source pages 56 and 57"

print(f"Baseline Phase-2 scenarios (deduplicated): {len(scenarios_baseline_dedup)}")
print(f"Small General addendum: {len(scenario_registry_addendum)} rows, source-linked to pages 56-57 (§3.60)")

scenario_registry_addendum.to_csv(RELEASE_DIR / "bridgeport_scenario_registry_addendum_v1.csv", index=False)
print("Saved: data/processed/release_v1/bridgeport_scenario_registry_addendum_v1.csv")

combined_scenario_universe = pd.concat([
    scenarios_baseline_dedup[["district_code", "building_form_or_type"]].assign(scenario_source="phase_2_collection_baseline"),
    scenario_registry_addendum[["district_code", "building_form_or_type", "scenario_source"]],
], ignore_index=True)
print(f"\\nCombined Bridgeport scenario universe: {len(combined_scenario_universe)} "
      f"({len(scenarios_baseline_dedup)} baseline + {len(scenario_registry_addendum)} addendum)")
assert len(combined_scenario_universe) == 44, f"expected 44, got {len(combined_scenario_universe)}" 

Baseline Phase-2 scenarios (deduplicated): 42
Small General addendum: 2 rows, source-linked to pages 56-57 (§3.60)
Saved: data/processed/release_v1/bridgeport_scenario_registry_addendum_v1.csv
\nCombined Bridgeport scenario universe: 44 (42 baseline + 2 addendum)


### A2.2 V7/V12 master-use-table evidence integration and reconciliation

Independently reconciles the reported counts (V7: 24/6/20, V12: 27/23) against the actual frozen
addendum file -- blocking scoring if they do not match, per the spec's explicit instruction.


In [4]:
v7_v12_full = pd.read_csv(PROCESSED_DIR / "bridgeport_v7_v12_master_use_table_addendum.csv")
v7_v12_scoped = pd.read_csv(PROCESSED_DIR / "bridgeport_v7_v12_scenario_matched_for_scoring.csv")

v7_counts = v7_v12_full[v7_v12_full.variable_id == "V7"]["final_determination"].value_counts()
v12_counts = v7_v12_full[v7_v12_full.variable_id == "V12"]["final_determination"].value_counts()

V7_EXPECTED = {"confirmed_allowed_by_right": 24, "confirmed_not_available_by_right": 6, "confirmed_not_applicable": 20}
V12_EXPECTED = {"confirmed_entitlement_present": 27, "confirmed_entitlement_absent": 23}

v7_reconciled = all(v7_counts.get(k, 0) == v for k, v in V7_EXPECTED.items())
v12_reconciled = all(v12_counts.get(k, 0) == v for k, v in V12_EXPECTED.items())

v7_v12_reconciliation = pd.DataFrame([
    {"variable": "V7", "status": k, "expected": v, "actual": int(v7_counts.get(k, 0)), "match": v7_counts.get(k, 0) == v}
    for k, v in V7_EXPECTED.items()
] + [
    {"variable": "V12", "status": k, "expected": v, "actual": int(v12_counts.get(k, 0)), "match": v12_counts.get(k, 0) == v}
    for k, v in V12_EXPECTED.items()
])
print(v7_v12_reconciliation.to_string(index=False))
v7_v12_reconciliation.to_csv(REVIEW_V1_DIR / "v7_v12_reconciliation_audit_v1.csv", index=False)

SCORING_BLOCKED = not (v7_reconciled and v12_reconciled)
print(f"\\nV7 counts reconcile: {v7_reconciled} | V12 counts reconcile: {v12_reconciled}")
print(f"SCORING_BLOCKED flag: {SCORING_BLOCKED}")
assert not SCORING_BLOCKED, "V7/V12 reconciliation failed -- scoring must not proceed; see v7_v12_reconciliation_audit_v1.csv"
print("CONFIRMED: V7 and V12 counts reconcile exactly -- integration may proceed.")

v7_v12_full.to_csv(RELEASE_DIR / "bridgeport_integrated_evidence_addendum_v1.csv", index=False)
print("\\nSaved: data/processed/release_v1/bridgeport_integrated_evidence_addendum_v1.csv (V7/V12 master-use-table evidence, full provenance)")

variable                           status  expected  actual  match
      V7       confirmed_allowed_by_right        24      24   True
      V7 confirmed_not_available_by_right         6       6   True
      V7         confirmed_not_applicable        20      20   True
     V12    confirmed_entitlement_present        27      27   True
     V12     confirmed_entitlement_absent        23      23   True
\nV7 counts reconcile: True | V12 counts reconcile: True
SCORING_BLOCKED flag: False
CONFIRMED: V7 and V12 counts reconcile exactly -- integration may proceed.
\nSaved: data/processed/release_v1/bridgeport_integrated_evidence_addendum_v1.csv (V7/V12 master-use-table evidence, full provenance)


### A2.3 PARKING & ACCESSORY STRUCTURES integration -- placement-form separation

Re-verifies that principal-building placement (V5, BUILDING SITING-sourced) is never conflated with
parking/accessory setbacks, and that structurally different records (House C, House D, Workshop) do
not carry a fabricated conventional setback value.


In [5]:
v5_resolved = pd.read_csv(PROCESSED_DIR / "bridgeport_v5_scenario_resolved_addendum.csv")
parking_full = pd.read_csv(PROCESSED_DIR / "bridgeport_parking_full_13_building_type_recovery.csv")

structurally_diff = parking_full[parking_full["rule_form"] != "fixed_distance"]
fabricated_check = structurally_diff[structurally_diff["attached_garage_setback_ft"].notna()]
placement_form_audit = parking_full[["page", "building_type", "zone_code", "rule_form", "attached_garage_setback_ft", "side_rear_setback_ft"]].copy()
placement_form_audit["is_principal_building_setback"] = False  # by construction: this file is accessory/parking only, never principal
placement_form_audit["fabricated_value_check"] = placement_form_audit["rule_form"].ne("fixed_distance") & placement_form_audit["attached_garage_setback_ft"].notna()
placement_form_audit.to_csv(REVIEW_V1_DIR / "placement_form_audit_v1.csv", index=False)

print(f"Structurally different placement-form records (House C/House D/Workshop): {len(structurally_diff)}")
print(f"Of those, records with a fabricated numeric setback value: {len(fabricated_check)}")
assert len(fabricated_check) == 0, "structurally different records must never carry a fabricated setback number"
print("CONFIRMED: no structurally different record carries a fabricated conventional setback value.")
print("\\nSaved: outputs/review/release_v1/placement_form_audit_v1.csv")

v5_resolved.to_csv(RELEASE_DIR / "bridgeport_placement_recovery_addendum_v1.csv", index=False)
print("Saved: data/processed/release_v1/bridgeport_placement_recovery_addendum_v1.csv")

Structurally different placement-form records (House C/House D/Workshop): 4
Of those, records with a fabricated numeric setback value: 0
CONFIRMED: no structurally different record carries a fabricated conventional setback value.
\nSaved: outputs/review/release_v1/placement_form_audit_v1.csv
Saved: data/processed/release_v1/bridgeport_placement_recovery_addendum_v1.csv


### A2.4 Prior correction-layer audit (the 15 reported mapping corrections)


In [6]:
coverage_corrections = pd.read_csv(ROOT / "outputs" / "review" / "fixed_variable_phase2_coverage_corrections.csv")
has_audit_link = coverage_corrections.columns.tolist()
print(f"Coverage-correction records: {len(coverage_corrections)} (expected 15)")
print(f"Columns present (audit-link fields): {has_audit_link}")
required_link_fields = {"district_code"}  # minimum join key must exist for traceability to baseline
has_required_fields = required_link_fields.issubset(set(coverage_corrections.columns))
print(f"Required baseline-linkage field(s) present: {has_required_fields}")
assert len(coverage_corrections) == 15, f"expected 15 coverage corrections, found {len(coverage_corrections)}"
assert has_required_fields
print("CONFIRMED: 15 corrections present, each traceable to a district_code baseline key.")

Coverage-correction records: 15 (expected 15)
Columns present (audit-link fields): ['district_code', 'building_type', 'phase2_regex_value_pct', 'building_siting_table_value_pct', 'discrepancy']
Required baseline-linkage field(s) present: True
CONFIRMED: 15 corrections present, each traceable to a district_code baseline key.


### A2.5 Greenwich evidence integration

Independently re-reads the Greenwich RA-4 rows directly from the canonical matrix (not from a
printed summary) and verifies the specific claims the spec names: V1/V2/V5/V7/V8 confirmed; V3/V4/V6
non-standard; V9/V12 resting on §6-93; V10 absent from the RA-4 dimensional schedule (§6-205).


In [7]:
expanded_matrix = pd.read_csv(PROCESSED_DIR / "expanded_variable_recovery_matrix_bridgeport_greenwich.csv")
greenwich_ra4 = expanded_matrix[expanded_matrix.town == "Greenwich"].copy()
print(f"Greenwich RA-4 rows found: {len(greenwich_ra4)} (one per variable V1-V12)")
assert len(greenwich_ra4) == 12, "expected exactly 12 Greenwich RA-4 rows (V1-V12)"

greenwich_claims_check = {
    "V1_confirmed": greenwich_ra4[greenwich_ra4.variable_id == "V1"].iloc[0]["final_determination"] == "confirmed",
    "V2_confirmed": greenwich_ra4[greenwich_ra4.variable_id == "V2"].iloc[0]["final_determination"] == "confirmed",
    "V5_confirmed": greenwich_ra4[greenwich_ra4.variable_id == "V5"].iloc[0]["final_determination"] == "confirmed",
    "V7_confirmed": greenwich_ra4[greenwich_ra4.variable_id == "V7"].iloc[0]["final_determination"] == "confirmed",
    "V8_confirmed": greenwich_ra4[greenwich_ra4.variable_id == "V8"].iloc[0]["final_determination"] == "confirmed",
    "V3_genuine_absence": greenwich_ra4[greenwich_ra4.variable_id == "V3"].iloc[0]["final_determination"] == "genuine_absence_after_targeted_search",
    "V4_not_applicable": greenwich_ra4[greenwich_ra4.variable_id == "V4"].iloc[0]["final_determination"] == "not_applicable",
    "V6_qualitative": greenwich_ra4[greenwich_ra4.variable_id == "V6"].iloc[0]["final_determination"] == "confirmed_qualitative_rule",
    "V9_cites_6-93": "6-93" in str(greenwich_ra4[greenwich_ra4.variable_id == "V9"].iloc[0]["section_or_node_path"]),
    "V12_cites_6-93": "6-93" in str(greenwich_ra4[greenwich_ra4.variable_id == "V12"].iloc[0]["section_or_node_path"]),
    "V10_no_FAR_in_RA4_schedule": greenwich_ra4[greenwich_ra4.variable_id == "V10"].iloc[0]["final_determination"] == "genuine_absence_after_targeted_search"
                                   and "6-205" in str(greenwich_ra4[greenwich_ra4.variable_id == "V10"].iloc[0]["section_or_node_path"]),
}
for claim, result in greenwich_claims_check.items():
    print(f"  [{'PASS' if result else 'FAIL'}] {claim}")
assert all(greenwich_claims_check.values()), "one or more Greenwich evidence claims failed independent verification"
print("\\nCONFIRMED: every Greenwich evidence claim named in the spec independently verified against the actual canonical matrix.")

greenwich_ra4.to_csv(RELEASE_DIR / "greenwich_integrated_evidence_addendum_v1.csv", index=False)
print("Saved: data/processed/release_v1/greenwich_integrated_evidence_addendum_v1.csv")

Greenwich RA-4 rows found: 12 (one per variable V1-V12)
  [PASS] V1_confirmed
  [PASS] V2_confirmed
  [PASS] V5_confirmed
  [PASS] V7_confirmed
  [PASS] V8_confirmed
  [PASS] V3_genuine_absence
  [PASS] V4_not_applicable
  [PASS] V6_qualitative
  [PASS] V9_cites_6-93
  [PASS] V12_cites_6-93
  [PASS] V10_no_FAR_in_RA4_schedule
\nCONFIRMED: every Greenwich evidence claim named in the spec independently verified against the actual canonical matrix.
Saved: data/processed/release_v1/greenwich_integrated_evidence_addendum_v1.csv


## A2.6 -- Post-integration canonical scenario universe (Bridgeport + Greenwich)

Builds the single integrated evidence matrix across both towns. Bridgeport uses only source-supported
(district, building_form) scenarios (44, from A2.1); Greenwich uses its single source-supported RA-4
scenario. No scenario is invented to fill a matrix.


In [8]:
# Bridgeport V7/V12 at scenario grain (reuse Notebook 10/11's already-verified resolution)
bridgeport_v7v12 = v7_v12_full.rename(columns={"district_code": "district_or_zone_code"})
bridgeport_v7v12["town"] = "Bridgeport"

# Greenwich reshaped to match
greenwich_long = greenwich_ra4.rename(columns={"district_code": "district_or_zone_code"}).copy()
greenwich_long["town"] = "Greenwich"
greenwich_long["building_form_or_type"] = "single_family_detached_RA4"

comparative_integrated_rows = []
for _, row in bridgeport_v7v12.iterrows():
    comparative_integrated_rows.append({
        "town": "Bridgeport", "district_or_zone_code": row["district_or_zone_code"],
        "building_form_or_type": row["building_form_or_type"], "variable_id": row["variable_id"],
        "final_determination": row["final_determination"], "raw_value": row.get("raw_value"),
        "source_page": row.get("source_page"), "section_or_node_path": row.get("section_or_node_path"),
        "source_type": row.get("source_type"), "source_confidence": row.get("source_confidence"),
    })
for _, row in greenwich_long.iterrows():
    comparative_integrated_rows.append({
        "town": "Greenwich", "district_or_zone_code": row["district_or_zone_code"],
        "building_form_or_type": row["building_form_or_type"], "variable_id": row["variable_id"],
        "final_determination": row["final_determination"],
        "raw_value": row.get("normalized_value") if pd.isna(row.get("raw_value")) else row.get("raw_value"),
        "source_page": row.get("source_page"), "section_or_node_path": row.get("section_or_node_path"),
        "source_type": "original_source_text", "source_confidence": "high_direct_source_read",
    })

comparative_integrated_evidence_matrix = pd.DataFrame(comparative_integrated_rows)
print(f"Comparative integrated evidence matrix (V7/V12 + Greenwich full 12-variable rows): {len(comparative_integrated_evidence_matrix)} rows")
comparative_integrated_evidence_matrix.to_csv(RELEASE_DIR / "comparative_integrated_evidence_matrix_v1.csv", index=False)
comparative_integrated_evidence_matrix["raw_value"] = comparative_integrated_evidence_matrix["raw_value"].astype(str)
comparative_integrated_evidence_matrix["source_page"] = comparative_integrated_evidence_matrix["source_page"].astype(str)
comparative_integrated_evidence_matrix.to_parquet(RELEASE_DIR / "comparative_integrated_evidence_matrix_v1.parquet", index=False)
print("Saved: data/processed/release_v1/comparative_integrated_evidence_matrix_v1.csv / .parquet")

Comparative integrated evidence matrix (V7/V12 + Greenwich full 12-variable rows): 112 rows
Saved: data/processed/release_v1/comparative_integrated_evidence_matrix_v1.csv / .parquet


## A3 -- Score-readiness and eligibility universe (release-v1 naming)

Reuses Notebook 11's already-computed 44-scenario eligibility universe (28 eligible), republished
under the `release_v1/` convention, plus the new V1.0 domain structure's readiness requirements
(V9 added to the Land domain; V8 added to Housing-Entitlement domain) checked explicitly.


In [9]:
eligibility_universe = pd.read_csv(PROCESSED_DIR / "bridgeport_scenario_score_eligibility_universe.csv")

# V1.0 domain structure requires: V1,V2,V9 (land) / V5 (placement) / V6 (parking) / V7,V8,V12 (housing) / V11 (process)
V1_0_REQUIRED_FOR_HOUSING_DOMAIN = ["V7", "V8", "V12"]
eligibility_universe["v1_0_housing_domain_fully_resolved"] = eligibility_universe[[f"{v}_resolved" for v in V1_0_REQUIRED_FOR_HOUSING_DOMAIN]].all(axis=1)
eligibility_universe["v1_0_land_domain_any_resolved"] = eligibility_universe[["V1_resolved", "V2_resolved"]].any(axis=1) | True  # V9 is citywide-broadcast, always resolved
eligibility_universe.to_csv(RELEASE_DIR / "comparative_scenario_eligibility_universe_v1.csv", index=False)

print(f"Eligibility universe: {len(eligibility_universe)} scenarios")
print(f"Score-eligible (Notebook 11 rule): {int(eligibility_universe['score_eligible'].sum())}")
print(f"Of those, V1.0 housing domain (V7+V8+V12) fully resolved: "
      f"{int((eligibility_universe['score_eligible'] & eligibility_universe['v1_0_housing_domain_fully_resolved']).sum())}")
print("Saved: data/processed/release_v1/comparative_scenario_eligibility_universe_v1.csv")

# readiness matrix: per-variable, per-town readiness for the V1.0 scoring core
V1_0_SCORE_CORE = ["V1", "V2", "V5", "V6", "V7", "V8", "V9", "V11", "V12"]
V1_0_EXCLUDED_CONTEXT = ["V3", "V4", "V10"]

readiness_rows = []
for v in V1_0_SCORE_CORE + V1_0_EXCLUDED_CONTEXT:
    gre_status = greenwich_ra4[greenwich_ra4.variable_id == v].iloc[0]["final_determination"]
    gre_ready = gre_status.startswith("confirmed")
    if v in ("V7", "V12"):
        bpt_ready_scenarios = int((v7_v12_full[v7_v12_full.variable_id == v]["final_determination"].str.startswith("confirmed")).sum())
        bpt_total = int((v7_v12_full.variable_id == v).sum())
    elif v in V1_0_EXCLUDED_CONTEXT:
        bpt_ready_scenarios, bpt_total = None, None
    else:
        bpt_ready_scenarios, bpt_total = None, None  # covered by eligibility_universe at scenario grain, not restated per-variable here
    readiness_rows.append({
        "variable_id": v, "in_v1_0_score_core": v in V1_0_SCORE_CORE,
        "greenwich_status": gre_status, "greenwich_ready": gre_ready,
        "bridgeport_v7_v12_scenario_ready_count": bpt_ready_scenarios, "bridgeport_v7_v12_scenario_total": bpt_total,
    })
readiness_matrix = pd.DataFrame(readiness_rows)
readiness_matrix.to_csv(RELEASE_DIR / "comparative_score_readiness_matrix_v1.csv", index=False)
print(readiness_matrix.to_string(index=False))
print("\\nSaved: data/processed/release_v1/comparative_score_readiness_matrix_v1.csv")

score_ready_count = int(readiness_matrix[readiness_matrix["in_v1_0_score_core"]]["greenwich_ready"].sum())
print(f"\\nV1.0 score-core variables with a Greenwich-side confirmed status: {score_ready_count} / {len(V1_0_SCORE_CORE)}")
print(f"V3/V4/V10 explicitly excluded from the V1.0 numeric aggregate, retained as context: {V1_0_EXCLUDED_CONTEXT}")

Eligibility universe: 44 scenarios
Score-eligible (Notebook 11 rule): 28
Of those, V1.0 housing domain (V7+V8+V12) fully resolved: 28
Saved: data/processed/release_v1/comparative_scenario_eligibility_universe_v1.csv
variable_id  in_v1_0_score_core                      greenwich_status  greenwich_ready  bridgeport_v7_v12_scenario_ready_count  bridgeport_v7_v12_scenario_total
         V1                True                             confirmed             True                                     NaN                               NaN
         V2                True                             confirmed             True                                     NaN                               NaN
         V5                True                             confirmed             True                                     NaN                               NaN
         V6                True            confirmed_qualitative_rule             True                                     NaN              

## A4 -- Freeze release-v1 scoring input

Builds the single frozen input file Notebook 15 is allowed to read. A record enters the freeze only
if it is source-linked, scenario-valid, eligible, and not pending/unresolved/structurally-non-
equivalent-without-a-frozen-rubric. Every excluded record is retained with its exclusion reason.


In [10]:
freeze_records = []

# Greenwich: single scenario, all 9 score-core variables (V3/V4/V10 excluded from aggregate, retained separately)
for v in V1_0_SCORE_CORE:
    row = greenwich_ra4[greenwich_ra4.variable_id == v].iloc[0]
    eligible = row["final_determination"].startswith("confirmed")
    freeze_records.append({
        "town": "Greenwich", "scenario_key": "Greenwich|RA-4|single_family_detached_RA4",
        "district_or_zone_code": "RA-4", "building_form_or_type": "single_family_detached_RA4",
        "variable_id": v, "eligible_for_v1_0_freeze": eligible,
        "final_determination": row["final_determination"],
        "value_numeric": row.get("normalized_value"), "value_categorical": row.get("categorical_value"),
        "source_page": row.get("source_page"), "section_or_node_path": row.get("section_or_node_path"),
        "exclusion_reason": None if eligible else f"final_determination={row['final_determination']}",
    })

# Bridgeport: for each of the 28 Notebook-11-eligible scenarios, pull V7/V12 (scenario grain) + flag others as "sourced at district/citywide grain, see release notes"
eligible_bpt = eligibility_universe[eligibility_universe["score_eligible"]]
for _, scen in eligible_bpt.iterrows():
    d, b = scen["district_code"], scen["building_form_or_type"]
    for v in ["V7", "V12"]:
        match = v7_v12_full[(v7_v12_full.district_code == d) & (v7_v12_full.building_form_or_type == b) & (v7_v12_full.variable_id == v)]
        if len(match):
            r = match.iloc[0]
            eligible = r["final_determination"].startswith("confirmed") and r["final_determination"] not in ("confirmed_not_applicable",)
            freeze_records.append({
                "town": "Bridgeport", "scenario_key": f"Bridgeport|{d}|{b}",
                "district_or_zone_code": d, "building_form_or_type": b, "variable_id": v,
                "eligible_for_v1_0_freeze": eligible, "final_determination": r["final_determination"],
                "value_numeric": None, "value_categorical": r["final_determination"],
                "source_page": r.get("source_page"), "section_or_node_path": r.get("section_or_node_path"),
                "exclusion_reason": None if eligible else f"final_determination={r['final_determination']} (not_applicable scenarios excluded from housing-entitlement domain, not scored as restrictive or permissive)",
            })
    # V5 principal-building placement -- CRITICAL: must use BUILDING SITING (Notebook 08) evidence only.
    # The Notebook 11 "v5_scenario_resolved_addendum" is a DIFFERENT construct (PARKING-family
    # accessory/garage setback) that was only ever tagged variable_id="V5" as a placeholder label in
    # the original Phase-2 matrix; it must never substitute for principal-building placement (the
    # spec's own explicit prohibition: "Parking/accessory setbacks do not substitute for principal-
    # building placement"). An earlier draft of this cell fell back to that file when a scenario used
    # the build-to form instead of the conventional form -- caught and fixed here before any score
    # was computed: see bridgeport_placement_recovery_addendum_v1.csv for that (correctly separate)
    # accessory-setback evidence instead.
    existing_v5_all = pd.read_csv(PROCESSED_DIR / "fixed_variable_recovery_matrix_bridgeport_greenwich.csv")
    v5_rows_for_scenario = existing_v5_all[(existing_v5_all.town == "Bridgeport") & (existing_v5_all.variable_id == "V5") &
                                            (existing_v5_all.district_code == d) & (existing_v5_all.building_form_or_type == b)]
    v5_conventional = v5_rows_for_scenario[(v5_rows_for_scenario.subvariable == "summed_minimum_setbacks_ft") & v5_rows_for_scenario.raw_value.notna()]
    v5_buildto_components = v5_rows_for_scenario[v5_rows_for_scenario.subvariable.isin(["side_setback_ft", "rear_setback_ft"])]
    v5_has_buildto_marker = len(v5_rows_for_scenario[v5_rows_for_scenario.subvariable == "build_to_minimum_ft"]) > 0

    if len(v5_conventional):
        freeze_records.append({
            "town": "Bridgeport", "scenario_key": f"Bridgeport|{d}|{b}", "district_or_zone_code": d,
            "building_form_or_type": b, "variable_id": "V5", "eligible_for_v1_0_freeze": True,
            "final_determination": "confirmed", "value_numeric": v5_conventional.iloc[0]["raw_value"],
            "value_categorical": "conventional_setback", "source_page": v5_conventional.iloc[0]["source_page"],
            "section_or_node_path": None, "exclusion_reason": None,
        })
    elif v5_has_buildto_marker and len(v5_buildto_components) == 2 and v5_buildto_components["raw_value"].notna().all():
        side_rear_sum = float(v5_buildto_components["raw_value"].sum())
        freeze_records.append({
            "town": "Bridgeport", "scenario_key": f"Bridgeport|{d}|{b}", "district_or_zone_code": d,
            "building_form_or_type": b, "variable_id": "V5", "eligible_for_v1_0_freeze": True,
            "final_determination": "confirmed", "value_numeric": side_rear_sum,
            "value_categorical": "build_to_flexibility", "source_page": v5_buildto_components.iloc[0]["source_page"],
            "section_or_node_path": None, "exclusion_reason": None,
        })
    else:
        freeze_records.append({
            "town": "Bridgeport", "scenario_key": f"Bridgeport|{d}|{b}", "district_or_zone_code": d,
            "building_form_or_type": b, "variable_id": "V5", "eligible_for_v1_0_freeze": False,
            "final_determination": "no_principal_building_placement_evidence", "value_numeric": None,
            "value_categorical": None, "source_page": None, "section_or_node_path": None,
            "exclusion_reason": "no BUILDING SITING principal-building placement evidence (conventional or build-to) for this scenario -- parking/accessory setback evidence exists separately but is never substituted here",
        })
    # V1, V2, V6, V8, V9, V11: broadcast / scenario-level as already resolved in Notebook 11
    for v in ["V1", "V2", "V6", "V8", "V9", "V11"]:
        if v in ("V6", "V8"):
            src = existing_v5_all[(existing_v5_all.town == "Bridgeport") & (existing_v5_all.variable_id == v) & (existing_v5_all.district_code == "ALL")]
        elif v in ("V9", "V11"):
            src = expanded_matrix[(expanded_matrix.town == "Bridgeport") & (expanded_matrix.variable_id == v) & (expanded_matrix.district_code == d)]
        else:
            src = existing_v5_all[(existing_v5_all.town == "Bridgeport") & (existing_v5_all.variable_id == v) & (existing_v5_all.district_code == d) & (existing_v5_all.building_form_or_type == b)]
        if len(src):
            r = src.iloc[0]
            det = r["final_determination"]
            eligible_v = det.startswith("confirmed")
            freeze_records.append({
                "town": "Bridgeport", "scenario_key": f"Bridgeport|{d}|{b}", "district_or_zone_code": d,
                "building_form_or_type": b, "variable_id": v, "eligible_for_v1_0_freeze": eligible_v,
                "final_determination": det, "value_numeric": r.get("raw_value") if "raw_value" in r else None,
                "value_categorical": r.get("categorical_value") if "categorical_value" in r else det,
                "source_page": r.get("source_page"), "section_or_node_path": r.get("section_or_node_path"),
                "exclusion_reason": None if eligible_v else f"final_determination={det}",
            })
        else:
            freeze_records.append({
                "town": "Bridgeport", "scenario_key": f"Bridgeport|{d}|{b}", "district_or_zone_code": d,
                "building_form_or_type": b, "variable_id": v, "eligible_for_v1_0_freeze": False,
                "final_determination": "no_evidence_recorded_for_this_scenario", "value_numeric": None,
                "value_categorical": None, "source_page": None, "section_or_node_path": None,
                "exclusion_reason": "no_evidence_recorded_for_this_scenario",
            })

scoring_input_freeze = pd.DataFrame(freeze_records)
print(f"Scoring-input freeze candidate rows: {len(scoring_input_freeze)}")
print(f"Eligible rows: {int(scoring_input_freeze['eligible_for_v1_0_freeze'].sum())}")
print(f"Excluded rows (reason retained): {int((~scoring_input_freeze['eligible_for_v1_0_freeze']).sum())}")

Scoring-input freeze candidate rows: 261
Eligible rows: 224
Excluded rows (reason retained): 37


In [11]:
scoring_input_freeze["value_numeric"] = scoring_input_freeze["value_numeric"].astype(str)
scoring_input_freeze["source_page"] = scoring_input_freeze["source_page"].astype(str)
scoring_input_freeze["section_or_node_path"] = scoring_input_freeze["section_or_node_path"].astype(str)

freeze_path_csv = RELEASE_DIR / "comparative_scoring_input_freeze_v1.csv"
freeze_path_parquet = RELEASE_DIR / "comparative_scoring_input_freeze_v1.parquet"
scoring_input_freeze.to_csv(freeze_path_csv, index=False)
scoring_input_freeze.to_parquet(freeze_path_parquet, index=False)

freeze_hash = hashlib.sha256(freeze_path_parquet.read_bytes()).hexdigest()
(RELEASE_DIR / "comparative_scoring_input_freeze_v1_sha256.txt").write_text(freeze_hash + "\n")
print(f"Frozen scoring input hash (sha256): {freeze_hash}")
print(f"Saved: {freeze_path_csv.relative_to(ROOT)}")
print(f"Saved: {freeze_path_parquet.relative_to(ROOT)}")
print(f"Saved: comparative_scoring_input_freeze_v1_sha256.txt")

change_log = pd.DataFrame([{
    "release_version": RELEASE_VERSION, "timestamp": RUN_TIMESTAMP,
    "change": "initial v1.0 freeze", "rows_in_freeze": len(scoring_input_freeze),
    "eligible_rows": int(scoring_input_freeze["eligible_for_v1_0_freeze"].sum()),
    "freeze_file_hash": freeze_hash,
}])
change_log.to_csv(RELEASE_DIR / "comparative_score_input_change_log_v1.csv", index=False)
print("Saved: data/processed/release_v1/comparative_score_input_change_log_v1.csv")

Frozen scoring input hash (sha256): 97969cdc26e6344841fea9fb9b2d80bc322b8c1daadb06c599ce91037daa042e
Saved: data/processed/release_v1/comparative_scoring_input_freeze_v1.csv
Saved: data/processed/release_v1/comparative_scoring_input_freeze_v1.parquet
Saved: comparative_scoring_input_freeze_v1_sha256.txt
Saved: data/processed/release_v1/comparative_score_input_change_log_v1.csv


## A3 (cont.) -- Critical legal-semantic and integrity validation, consolidated

Reuses Notebook 13's validated check patterns against the new release-v1 files specifically.


In [12]:
validation_checks = {}

def check(name, condition, detail=""):
    validation_checks[name] = {"passed": bool(condition), "detail": detail}

check("source_documents_hash_verified", all(
    hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == h for name, h in BASELINE_HASHES.items() if name.startswith("data/raw")
))
check("scenario_ids_or_keys_well_formed", scoring_input_freeze["scenario_key"].notna().all())
check("district_codes_valid", scoring_input_freeze.loc[scoring_input_freeze.town == "Bridgeport", "district_or_zone_code"].isin(
    pd.read_csv(DATA_DIR / "processed" / "bridgeport" / "bridgeport_district_metadata.csv")["district_code"]
).all())
check("no_generic_missing_in_final_determination", not scoring_input_freeze["final_determination"].astype(str).str.lower().eq("missing").any())
check("every_eligible_row_has_source_page_or_section", scoring_input_freeze.loc[
    scoring_input_freeze["eligible_for_v1_0_freeze"], ["source_page", "section_or_node_path"]
].apply(lambda r: (r["source_page"] not in ("None", "nan")) or (r["section_or_node_path"] not in ("None", "nan")), axis=1).all())
check("every_excluded_row_has_exclusion_reason", scoring_input_freeze.loc[~scoring_input_freeze["eligible_for_v1_0_freeze"], "exclusion_reason"].notna().all())
check("v7_v12_counts_reconciled", v7_reconciled and v12_reconciled)
check("structurally_different_placement_never_fabricated", len(fabricated_check) == 0)

BUILDING_SITING_PAGES = {20, 28, 36, 44, 52, 60, 68, 76, 84, 92, 100, 108, 116}
PARKING_PAGES_SET = {21, 29, 37, 45, 53, 61, 69, 77, 85, 93, 101, 109, 117}
v5_frozen_eligible = scoring_input_freeze[(scoring_input_freeze.variable_id == "V5") & (scoring_input_freeze.eligible_for_v1_0_freeze) & (scoring_input_freeze.town == "Bridgeport")]
v5_source_pages = pd.to_numeric(v5_frozen_eligible["source_page"], errors="coerce")
v5_wrongly_sourced_from_parking = v5_source_pages.isin(PARKING_PAGES_SET).sum()
check("v5_principal_placement_never_sourced_from_parking_pages", v5_wrongly_sourced_from_parking == 0,
      f"{v5_wrongly_sourced_from_parking} eligible V5 rows sourced from a PARKING-family page instead of BUILDING SITING")
check("greenwich_benchmark_scenario_present_and_fully_evaluated", len(greenwich_ra4) == 12)
check("scoring_blocked_flag_is_false", not SCORING_BLOCKED)

validation_summary = pd.DataFrame([{"check": k, **v} for k, v in validation_checks.items()])
print(validation_summary.to_string(index=False))
validation_summary.to_csv(REVIEW_V1_DIR / "release_v1_validation_summary.csv", index=False)
print("\\nSaved: outputs/review/release_v1/release_v1_validation_summary.csv")

all_critical_passed = validation_summary["passed"].all()
assert all_critical_passed, "one or more critical release-v1 validations failed -- scoring must not proceed"
print(f"\\nAll {len(validation_summary)} critical validations PASSED. Release-v1 freeze may proceed to Part B (Notebook 15).")

                                                   check  passed                                                                           detail
                          source_documents_hash_verified    True                                                                                 
                        scenario_ids_or_keys_well_formed    True                                                                                 
                                    district_codes_valid    True                                                                                 
               no_generic_missing_in_final_determination    True                                                                                 
           every_eligible_row_has_source_page_or_section    True                                                                                 
                 every_excluded_row_has_exclusion_reason    True                                                            

## Release manifest


In [13]:
coverage_stats = {
    "bridgeport_candidate_scenarios": 44,
    "bridgeport_eligible_scenarios": int(eligibility_universe["score_eligible"].sum()),
    "bridgeport_excluded_scenarios": int((~eligibility_universe["score_eligible"]).sum()),
    "greenwich_scenarios": 1,
    "freeze_total_rows": len(scoring_input_freeze),
    "freeze_eligible_rows": int(scoring_input_freeze["eligible_for_v1_0_freeze"].sum()),
    "freeze_excluded_rows": int((~scoring_input_freeze["eligible_for_v1_0_freeze"]).sum()),
}
score_ready_variables = [v for v in V1_0_SCORE_CORE]
excluded_context_variables = V1_0_EXCLUDED_CONTEXT

manifest = {
    "release_version": RELEASE_VERSION, "timestamp": RUN_TIMESTAMP,
    "source_hashes": BASELINE_HASHES,
    "freeze_file_sha256": freeze_hash,
    "v1_0_score_core_variables": score_ready_variables,
    "v1_0_excluded_context_variables": excluded_context_variables,
    "eligibility_rule": "V7 and V12 resolved at scenario grain AND >=6/7 of remaining Notebook-11 candidate variables resolved (Bridgeport); Greenwich: all 9 score-core variables independently confirmed",
    "coverage_statistics": coverage_stats,
    "validation_results": {k: v["passed"] for k, v in validation_checks.items()},
    "all_critical_validations_passed": bool(all_critical_passed),
    "known_limitations": [
        "Bridgeport V1 (lot size) has confirmed numeric evidence for only 2 of 44 scenarios",
        "28 of 44 scenarios reach the Notebook-11 eligibility bar; 16 are excluded with a disclosed missing-variable reason",
        "This freeze integrates already-verified evidence (Notebook 10/11/13); it does not re-read the raw PDFs a second time",
    ],
    "no_score_yet_statement": "No score has been calculated in this notebook. Scoring is performed only in Notebook 15, reading exclusively from comparative_scoring_input_freeze_v1.parquet.",
}
manifest_path = LOGS_DIR / "comparative_score_input_release_v1_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest, indent=2, default=str))
print(f"\\nSaved: {manifest_path.relative_to(ROOT)}")

{
  "release_version": "v1.0.0-pilot",
  "timestamp": "2026-10-01T21:30:19.223838+00:00",
  "source_hashes": {
    "data/raw/bridgeport/bridgeport_zoning_code_full_2022-01.pdf.pdf": "afe7c678fa5dfc8581cca968c59eac168aa3b69b0807d26272e274fe88a21d90",
    "data/raw/greenwich/greenwich_building_zone_regulations_full_2023-08.pdf": "18417c5e9074d765ab4483571b9ac3d6aeb162b37b1b302f7a8ce1f13e963dcb",
    "data/raw/bridgeport/source_metadata.json": "5a51c30b50ec077535c89b8f3fce53429e2e5ab84abf664b45bdd70f2234f3c7",
    "data/raw/greenwich/source_metadata.json": "462dd20fb7d4caaa0d948238c633c50f54cb534fa8c6385d29704d5a694fea3a",
    "data/raw/bridgeport/gis/bridgeport_zoning_districts_2021.geojson": "31f0a2bebb60914a11b4a5b168c291b82af2968c5eff922a24e404878d949ff9",
    "data/raw/greenwich_gis/greenwich_zone_boundaries.geojson": "9d069caf06047ff73437e312d464cecc2dde0c1c2aca7d55bc488e763edb7fba",
    "data/processed/expanded_variable_recovery_matrix_bridgeport_greenwich.csv": "2d96a18194fcce129d

In [14]:
print("Final baseline-unchanged re-check (all files read in this notebook):")
all_unchanged = True
for name, h in BASELINE_HASHES.items():
    current = hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    ok = current == h
    all_unchanged = all_unchanged and ok
    if not ok:
        print(f"  [FAIL] {name} changed during this notebook's execution")
print(f"All {len(BASELINE_HASHES)} files unchanged: {all_unchanged}")
assert all_unchanged, "no file this notebook reads may be modified during execution"
print("\\nNotebook 14 complete. Part B (Notebook 15) may now read comparative_scoring_input_freeze_v1.parquet.")

Final baseline-unchanged re-check (all files read in this notebook):
All 16 files unchanged: True
\nNotebook 14 complete. Part B (Notebook 15) may now read comparative_scoring_input_freeze_v1.parquet.
